# Single Cell data
CC 2026-08-11

## 1. Setup

In [ ]:
import retinanalysis as ra
import matplotlib.pyplot as plt
import numpy as np

# Read-only single-cell database queries and notebook browsers.
from retinanalysis.SCutils import explore as sc

## 2. Map H5 files

Create JSON metadata for newly added single-cell H5 files.

In [ ]:
# Map new h5 files when needed.
report = ra.SCutils.update_single_cell_json()

## 3. Populate and refresh the database

`populate_database()` ingests new experiments, refreshes experiments whose JSON changed, and returns the database freshness check in the same report. Canonical experiment names such as `YYYY-MM-DD_X.h5` are included; auxiliary and legacy files are ignored. By default only metadata and tags JSON files trigger a refresh. Pass `watch_data_file=True` to include H5 modification times.

In [ ]:
# One call handles ingest, refresh, and the post-ingest stale-file check.
summary = ra.populate_database()
df_db = summary['experiments']
df_stale = summary['stale']

print(f"newly added : {len(summary['added'])}")
print(f"refreshed   : {len(summary['updated'])}")
print(f"errored     : {len(summary['skipped'])}")
print(f"database    : {len(df_db)} experiments; {len(df_stale)} still out of date")

if len(df_stale):
    display(df_stale[['exp_name', 'date_added', 'source_mtime', 'source_file']])

### 3.1 Purging entries

Deleting an experiment cascades through its animals, preparations, cells, epoch groups, blocks, epochs, and responses. These calls do not delete H5 or JSON files, so a later populate can restore the data. The destructive examples remain commented out.

In [ ]:
# ra.purge_experiments('2026-06-04_G')
# ra.purge_experiments(['2026-05-06_E', '2026-05-08_E'])

# Drop only rows that remain stale after populate.
# ra.purge_experiments(df_stale['exp_name'].tolist())

# Wipe the whole database (requires the literal confirmation token).
# ra.purge_database(confirm='YES_DELETE_ALL')

print(f'{len(df_db)} experiments currently in the database.')

## 4. List single-cell experiments

Experiments are grouped into `chris_data` and `fred_data`. Each row includes species, experiment name, the best available project label, and the cell types recorded that day.

In [ ]:
df_sc_exps = sc.list_experiments()

## 5. Find experiments by protocol

Search protocol names case-insensitively. The returned DataFrame remains one row per epoch block.

In [ ]:
df_blocks = sc.find_blocks('spotWithAnnularContrastReversingGrating')

## 6. Browse and summarize experiments

Use the cascading menus to select data owner, species, and experiment. The overview is organized as cell → epoch group (group label) → protocol, with block and epoch counts. Then select an epoch block and click **Load original traces** to read and display every unprocessed Amp1 epoch trace from the H5 file.

In [ ]:
experiment_browser = sc.summarize_experiments(df_sc_exps)

## 7. ExpandingSpots: raster, PSTH and area summation

Pick a cell and analyze its expanding-spots data. We require a **cell-attached** block on purpose — the analysis counts spikes, and whole-cell recordings have no extracellular spikes to detect. If the requested cell never ran the protocol, the analysis falls back to the first cell that did.

Three panels: the spike raster grouped by spot size, the Gaussian-kernel PSTH per spot size (port of `spikeTimeToPSTH.m`), and the area-summation curve fitted with a difference of Gaussians (`DoG.m` / `fitDoG.m`) to pull out the center and surround sigma.

In [ ]:
# Freeze the browser's current experiment selection for the analysis below.
exp_name = experiment_browser.selectors['experiment'].value
df_exp = experiment_browser.df_exp
cell_label = 'Cell1'
sc.protocol_tree(df_exp.query('cell_label == @cell_label'), height=300);

In [ ]:
# ExpandingSpots for one cell, end to end: resolve a cell-attached block (block
# ids are DB auto-increments, so never hardcode one), load it, detect spikes,
# count them in the stimulus window, fit the difference-of-Gaussians area
# summation model, and plot the raster + summation curve.
# Port of analyzeExpandingSpots.m; the model matches DoG.m / fitDoG.m.
from retinanalysis.SCutils.protocols import expanding_spots as es

# min_peak_amplitude is the amplitude floor for spike detection -- the MATLAB's
# thresholdSpikeFactor. It matters: with no floor, this recording's clustering
# calls ~600 low-amplitude noise peaks per epoch "spikes" and then discards the
# whole epoch, leaving 20/27 epochs empty and r2 = 0.25. Anything from 20 to 80
# recovers the same sigmas here (r2 = 0.98). analyze_expanding_spots warns when
# too many epochs come back silent.
res = es.analyze_expanding_spots(exp_name, cell_label=cell_label,
                                 detector_kwargs={'min_peak_amplitude': 40})
res.fit

### 7.1 Check the spike detection

Spike counts drive the whole area-summation fit, so eyeball a raw trace with the detected spikes marked. `i_epoch` indexes epochs in acquisition order.

In [ ]:
es.plot_example_trace(res, i_epoch=2);

## 8. Export to Igor

`ra.igor_export` is the Python counterpart of `makeAxisStructChris.m`: it walks the matplotlib artists and writes the same flat HDF5 the Igor procedures (`DisplayFigFromMatlab`) already load — one group named after the file, `<wave>_Y` / `<wave>_X` pairs, colors, marker numbers, errorbar deltas.

Two habits make the export clean: give artists a `label=` (it becomes the Igor wave name, sanitized) and keep numbers out of those labels so wave names stay stable across cells. Shaded spans, text and legends are decoration and are not exported.

Set `RA_IGOR_DIR` to your Igor project folder to write straight where Igor reads; otherwise files land in `<OUTPUT_DIR>/igor_h5`.

In [ ]:
# One .h5 per panel, as makeAxisStructChris.m writes one per axis. The group
# inside each file is named after the file stem, which Igor uses as the data
# folder name. res.fig is the figure drawn above, so nothing is redrawn here.
# Pass basedir=... to override RA_IGOR_DIR for a one-off.
paths = ra.igor_export.export_figure_to_h5(
    res.fig, f'expSpots_{exp_name}_{res.cell_label}'.replace('-', '_'))